In [ ]:
# CELL 0 - Environment Check
# ─────────────────────────────────────────────────────────────────────────────
# PrecisionSkin LESION DETECTOR v3 (same layout as the facial-only notebook).
# Run the cells ONE BY ONE, top to bottom. Kernel: the WSL  .venv_det  (PyTorch + ultralytics).
#
# What this notebook does:
#   1. takes the detector training data you already have (dataset v1),
#   2. adds YOUR hand-drawn boxes (from Roboflow or from lesion_box_tool.py) + Normal skin photos with no boxes,
#   3. trains a new YOLOv8n detector with the SAME recipe as v1,
#   4. compares v1 and v3 on the original test photos, on your own hand-boxed photos, and on healthy skin,
#   5. exports the ONNX file and the records. v1 stays in the app unless v3 is clearly better.
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, json, time, shutil, random, hashlib, datetime, platform
from pathlib import Path
import numpy as np, pandas as pd, yaml, cv2
import matplotlib.pyplot as plt
from PIL import Image

print("Python :", sys.version.split()[0])
print("Kernel :", sys.executable)
if ".venv_det" not in sys.executable:
    print("WARNING: this is not the .venv_det kernel; PyTorch/ultralytics may be missing.")
print("✓ imports OK")

In [ ]:
# CELL 1 - GPU / PyTorch Check
# ─────────────────────────────────────────────────────────────────────────────
# Confirms PyTorch can see your GPU before anything heavy runs.
# Without a GPU, training would take many hours (CPU mode is only for testing).
# ─────────────────────────────────────────────────────────────────────────────
import torch
from ultralytics import YOLO

print("PyTorch      :", torch.__version__)
print("CUDA ready   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected - training will be very slow.")

In [ ]:
# CELL 2 - Local Path Configuration
# ─────────────────────────────────────────────────────────────────────────────
# Every folder this notebook reads or writes. Edit ONLY this cell when a path differs.
#
#   OWN_BOX_SOURCES : your hand-drawn boxes. Each entry is ONE YOLO-format folder:
#       - exported from Roboflow ("YOLOv8" format: has data.yaml + train/valid/test with images/ and labels/)
#       - or a folder made by lesion_box_tool.py (images/ and labels/)
#     "name"   : a short name (used in file names)
#     "path"   : the folder
#     "splits" : which sub-folders to use for TRAINING (Roboflow: train, valid, test)
#     "replace_public_twins" : True = the SAME photo may already exist in the public datasets with other boxes. Your boxes then REPLACE
#                              the public ones (all copies of that photo are removed from training; a photo whose public copy sits in
#                              validation/test is NOT used, so nothing leaks into the test set).
#   OWN_TEST_DIR    : the hand-boxed TEST photos (never trained on): folder with images/ and labels/ (class 0/1/2 = acne/hyperpigmentation/eczema)
#   NORMAL_DIR      : healthy-skin photos (no lesions) - used as "no box" examples and for a false-alarm test
# ─────────────────────────────────────────────────────────────────────────────
HOME = Path.home() / "PrecisionSkinV1"
WORK = HOME / "detector_work"

V1_DATASET = WORK / "dataset_v1"                                                         # the data v1 was trained on
V1_MODEL = WORK / "runs" / "detector_v1_yolov8n_seed42" / "weights" / "best.pt"          # the detector now in the app
EXPORT_DIR = WORK / "export"

OWN_BOX_SOURCES = [
    # {"name": "eczema_hand", "path": Path("/mnt/c/Users/Bautista/Downloads/own_eczema.yolov8"), "splits": ["train", "valid", "test"]},
    # {"name": "hyper_hand",  "path": Path("/mnt/c/Users/Bautista/Downloads/own_hyperpigmentation.yolov8"), "splits": ["train", "valid", "test"]},
]
OWN_TEST_DIR = HOME / "detector_own_images" / "own_images_v3" / "own_test"               # made by section 06 of the "own images" notebook; set None to skip
_CLASSES_CANDIDATES = [Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Classes"), Path("/mnt/c/Users/Bautista/Downloads/Classes")]     # the first existing folder is used
NORMAL_DIR = next((p / "Normal" for p in _CLASSES_CANDIDATES if (p / "Normal").exists()), _CLASSES_CANDIDATES[0] / "Normal")

RUN_NAME = "detector_v3_own_boxes"
OUT_ROOT = HOME / "detector_v3" / RUN_NAME
for sub in ("dataset", "figures", "reports"):
    (OUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_ROOT)

In [ ]:
# CELL 3 - Global Configuration (Classes, Seeds, Hyperparameters)
# ─────────────────────────────────────────────────────────────────────────────
# The training recipe is the SAME as v1 so that v1 and v3 can be compared fairly.
#   CLASSES must keep this order (it is the class number inside every label file).
#   APP_THRESHOLDS are the confidence limits the app uses for each class (Settings page).
# ─────────────────────────────────────────────────────────────────────────────
CLASSES = ["acne", "hyperpigmentation", "eczema"]
CLASS_ID = {c: i for i, c in enumerate(CLASSES)}
APP_THRESHOLDS = {"acne": 0.20, "hyperpigmentation": 0.25, "eczema": 0.25}

SEED = 42
MODEL_START = "yolov8n.pt"          # COCO-pretrained starting weights (same as v1)
IMGSZ = 640
EPOCHS = 100
PATIENCE = 25                       # early stopping
BATCH = 16
WORKERS = 2                         # keep small: WSL has little RAM

OWN_REPEAT = 2                      # your hand-boxed training photos are used this many times per epoch (they are the best labels)
N_NORMAL_TRAIN = 1000               # healthy photos added to training with NO boxes
N_NORMAL_HOLDOUT = 200              # healthy photos kept OUT of training, used to count false alarms

random.seed(SEED); np.random.seed(SEED)
print("classes      :", CLASSES)
print("recipe       : %s | %d px | up to %d epochs (patience %d) | batch %d | seed %d" % (MODEL_START, IMGSZ, EPOCHS, PATIENCE, BATCH, SEED))
print("✓ configuration set")

In [ ]:
# CELL 4 - Verify Inputs Exist
# ─────────────────────────────────────────────────────────────────────────────
# Stops early with a clear message if something is missing.
# ─────────────────────────────────────────────────────────────────────────────
problems = []
for label, p in (("v1 model", V1_MODEL), ("v1 dataset", V1_DATASET / "train" / "images"), ("Normal photos", NORMAL_DIR)):
    ok = Path(p).exists()
    print("%-16s %s  %s" % (label, "✓" if ok else "✗ MISSING", p))
    if not ok: problems.append(label)
for s in OWN_BOX_SOURCES:
    ok = Path(s["path"]).exists()
    print("%-16s %s  %s" % ("own: " + s["name"], "✓" if ok else "✗ MISSING", s["path"]))
    if not ok: problems.append(s["name"])
if OWN_TEST_DIR:
    ok = (Path(OWN_TEST_DIR) / "images").exists()
    print("%-16s %s  %s" % ("own test", "✓" if ok else "✗ MISSING", OWN_TEST_DIR))
    if not ok: problems.append("own test")
if not OWN_BOX_SOURCES:
    print("\nNOTE: OWN_BOX_SOURCES is empty - the new model will be trained on v1 data + Normal photos only.")
assert not problems, "Fix these paths in CELL 2: %s" % problems
v1_model = YOLO(str(V1_MODEL))
assert [v1_model.names[i] for i in range(len(CLASSES))] == CLASSES, "v1 class order differs from CLASSES"
print("\n✓ all inputs found | v1 classes:", v1_model.names)

In [ ]:
# CELL 5 - Dataset Inventory (exact numbers you have right now)
# ─────────────────────────────────────────────────────────────────────────────
# Counts photos and boxes per class for: the v1 data (train / val / test), each of your own box folders and your hand-boxed test photos.
# Repeat copies (eczema x3 in v1 training) are listed separately so you see the UNIQUE numbers.
# ─────────────────────────────────────────────────────────────────────────────
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def count_labels(label_dir, skip_repeats=False):
    photos, boxes, empty = {c: 0 for c in CLASSES}, {c: 0 for c in CLASSES}, 0
    for f in Path(label_dir).glob("*.txt"):
        if skip_repeats and "_rep" in f.name:
            continue
        seen, n = set(), 0
        for ln in f.read_text().splitlines():
            t = ln.split()
            if len(t) >= 5 and int(float(t[0])) < len(CLASSES):
                c = CLASSES[int(float(t[0]))]; boxes[c] += 1; seen.add(c); n += 1
        for c in seen: photos[c] += 1
        empty += (n == 0)
    return photos, boxes, empty

rows = []
for sp in ("train", "val", "test"):
    ph, bx, em = count_labels(V1_DATASET / sp / "labels", skip_repeats=True)
    rows.append(dict(set="v1 " + sp + " (unique)", **{c + " photos": ph[c] for c in CLASSES}, **{c + " boxes": bx[c] for c in CLASSES}, no_box_files=em))
if OWN_TEST_DIR:
    ph, bx, em = count_labels(Path(OWN_TEST_DIR) / "labels")
    rows.append(dict(set="your hand-boxed TEST", **{c + " photos": ph[c] for c in CLASSES}, **{c + " boxes": bx[c] for c in CLASSES}, no_box_files=em))
inv = pd.DataFrame(rows).set_index("set")
print(inv.T.to_string())
print("\n(own box folders are counted after their class names are matched, in CELL 6)")

In [ ]:
# CELL 6 - Build Dataset v3
# ─────────────────────────────────────────────────────────────────────────────
# dataset/
#   train/  = v1 training files (linked, not copied)  +  your hand-boxed photos (copied, repeated OWN_REPEAT times)
#             + N_NORMAL_TRAIN healthy photos with EMPTY label files
#   val/    = exactly the v1 validation photos (linked)
#   test/   = exactly the v1 test photos (linked)   <- same photos as v1, so the comparison is fair
# Your original files and the v1 dataset are NEVER changed.
#
# Class names in your Roboflow export are matched to CLASSES by name (upper/lower case ignored).
# Polygons are turned into the box that encloses them.
# ─────────────────────────────────────────────────────────────────────────────
DS = OUT_ROOT / "dataset"
if DS.exists():
    shutil.rmtree(DS)
for sp in ("train", "val", "test"):
    (DS / sp / "images").mkdir(parents=True); (DS / sp / "labels").mkdir(parents=True)
    for f in (V1_DATASET / sp / "images").iterdir():
        os.symlink(f, DS / sp / "images" / f.name)
    for f in (V1_DATASET / sp / "labels").iterdir():
        os.symlink(f, DS / sp / "labels" / f.name)

def name_map(source_path):
    y = Path(source_path) / "data.yaml"
    if not y.exists():
        return None
    names = yaml.safe_load(open(y)).get("names", {})
    names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
    mp = {}
    for k, v in names.items():
        key = str(v).strip().lower()
        if key not in CLASS_ID:
            raise ValueError("Class '%s' in %s/data.yaml is not one of %s. Rename it in Roboflow or add a mapping." % (v, source_path, CLASSES))
        mp[k] = CLASS_ID[key]
    return mp

def find_image_label_dirs(src, splits):
    src = Path(src)
    if (src / "images").exists():
        yield src / "images", src / "labels"
    for sp in splits:
        if (src / sp / "images").exists():
            yield src / sp / "images", src / sp / "labels"

# ---- optional: your boxes REPLACE the public boxes of the same photo ----
def small_thumb(path):
    im = cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_GRAYSCALE)
    return None if im is None else cv2.resize(im, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)

def zscore(x):
    x = x.reshape(len(x), -1).astype(np.float32); x = x - x.mean(1, keepdims=True)
    return (x / (x.std(1, keepdims=True) + 1e-6) / np.sqrt(x.shape[1])).astype(np.float32)

V1_MAN = pd.read_csv(V1_DATASET / "dataset_manifest.csv")
removed_groups, twin_report = set(), []

def plan_twins(src, mp, thr=0.97):
    """Returns the set of your photos that must NOT be used (their public copy is in validation/test)."""
    own = []                                                           # (image path, set of condition names)
    for img_dir, lab_dir in find_image_label_dirs(src["path"], src.get("splits", ["train", "valid", "test"])):
        for img in img_dir.iterdir():
            lab = lab_dir / (img.stem + ".txt")
            if img.suffix.lower() in IMG_EXT and lab.exists():
                ks = {int(float(l.split()[0])) for l in lab.read_text().splitlines() if len(l.split()) >= 5}
                ks = {(mp[k] if mp else k) for k in ks}
                if ks:
                    own.append((img, {CLASSES[k] for k in ks}))
    leak = set()
    for cond in sorted({c for _, cs in own for c in cs}):
        mine = [(p, cs) for p, cs in own if cond in cs]
        pub = V1_MAN[V1_MAN.condition.str.lower() == cond].reset_index(drop=True)
        pub_t = [small_thumb(V1_DATASET / r.split / "images" / r.file) for r in pub.itertuples()]
        ok = [i for i, t in enumerate(pub_t) if t is not None]
        D = zscore(np.stack([pub_t[i] for i in ok]))
        own_t = [small_thumb(p) for p, _ in mine]
        keep = [i for i, t in enumerate(own_t) if t is not None]
        G = np.stack([own_t[i] for i in keep])
        best = np.zeros((len(keep), len(ok)), np.float32)
        for k in range(4):
            for flip in (False, True):
                v = np.rot90(G, k, axes=(1, 2)); v = v[:, :, ::-1] if flip else v
                best = np.maximum(best, zscore(np.ascontiguousarray(v)) @ D.T)
        n_twin = n_leak = 0
        for row, i in enumerate(keep):
            hit = np.where(best[row] >= thr)[0]
            if len(hit) == 0:
                continue
            rows = pub.iloc[[ok[j] for j in hit]]
            n_twin += 1
            if rows.split.isin(["val", "test"]).any():
                leak.add(mine[i][0]); n_leak += 1
            else:
                removed_groups.update(rows.duplicate_group.tolist())
        twin_report.append(dict(source=src["name"], condition=cond, your_photos=len(mine), with_public_copy=n_twin, skipped_public_copy_in_val_or_test=n_leak))
    return leak

n_added = {c: [0, 0] for c in CLASSES}                      # [photos, boxes]
for s in OWN_BOX_SOURCES:
    mp = name_map(s["path"])
    skip_imgs = plan_twins(s, mp) if s.get("replace_public_twins") else set()
    for img_dir, lab_dir in find_image_label_dirs(s["path"], s.get("splits", ["train", "valid", "test"])):
        for img in img_dir.iterdir():
            if img.suffix.lower() not in IMG_EXT or img in skip_imgs:
                continue
            lab = lab_dir / (img.stem + ".txt")
            out_lines, cls_seen = [], set()
            for ln in (lab.read_text().splitlines() if lab.exists() else []):
                t = ln.split()
                if len(t) < 5:
                    continue
                k = int(float(t[0])); k = mp[k] if mp else k
                nums = [float(v) for v in t[1:]]
                if len(nums) > 4:                                             # polygon -> enclosing box
                    xs, ys = nums[0::2], nums[1::2]
                    cx, cy, w, h = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
                else:
                    cx, cy, w, h = nums
                if w > 0.003 and h > 0.003:
                    out_lines.append("%d %.6f %.6f %.6f %.6f" % (k, cx, cy, w, h)); cls_seen.add(CLASSES[k]); n_added[CLASSES[k]][1] += 1
            if not out_lines:
                continue                                                      # photos without a box are not used here
            for c in cls_seen:
                n_added[c][0] += 1
            for rep in range(OWN_REPEAT):
                base = "own_%s_%s%s" % (s["name"], img.stem, "" if rep == 0 else "_rep%d" % rep)
                shutil.copy2(img, DS / "train" / "images" / (base + img.suffix.lower()))
                (DS / "train" / "labels" / (base + ".txt")).write_text("\n".join(out_lines) + "\n")

# remove the public copies (and their repeats) of every photo you re-boxed
n_removed = 0
if removed_groups:
    gone = V1_MAN[(V1_MAN.split == "train") & V1_MAN.duplicate_group.isin(removed_groups)]
    for r in gone.itertuples():
        stem = Path(r.file).stem
        for f in list((DS / "train" / "images").glob(stem + "*")) + list((DS / "train" / "labels").glob(stem + "*")):
            if f.stem == stem or f.stem.startswith(stem + "_rep"):
                f.unlink(); n_removed += 1
if twin_report:
    print("PUBLIC COPIES OF YOUR PHOTOS:"); print(pd.DataFrame(twin_report).to_string(index=False))
    print("files removed from v1 training because your boxes replace them: %d (duplicate groups: %d)" % (n_removed, len(removed_groups)))

# healthy photos: some for training (no boxes), some kept out for the false-alarm test
normal = sorted(p for p in NORMAL_DIR.iterdir() if p.suffix.lower() in IMG_EXT)
random.Random(SEED).shuffle(normal)
hold, trainbg = normal[:N_NORMAL_HOLDOUT], normal[N_NORMAL_HOLDOUT:N_NORMAL_HOLDOUT + N_NORMAL_TRAIN]
for p in trainbg:
    shutil.copy2(p, DS / "train" / "images" / ("normal_" + p.name))
    (DS / "train" / "labels" / ("normal_" + p.stem + ".txt")).write_text("")
HOLDOUT_NORMAL = [str(p) for p in hold]

yaml.safe_dump({"path": str(DS), "train": "train/images", "val": "val/images", "test": "test/images", "names": {i: c for i, c in enumerate(CLASSES)}}, open(DS / "data.yaml", "w"))
print("your hand-boxed photos added (unique):", {c: "%d photos / %d boxes" % tuple(v) for c, v in n_added.items()})
print("healthy photos with no boxes added  :", len(trainbg), "| kept out for the false-alarm test:", len(hold))
print("✓ dataset v3 built at", DS)

In [ ]:
# CELL 7 - Class Counts per Split
# ─────────────────────────────────────────────────────────────────────────────
# Final number of files and boxes per class in the dataset the model will train on.
# Check: val and test are the same size as v1 (566 each); train = v1 + your photos (x OWN_REPEAT) + healthy photos.
# ─────────────────────────────────────────────────────────────────────────────
rows = []
for sp in ("train", "val", "test"):
    ph, bx, em = count_labels(DS / sp / "labels")
    rows.append(dict(split=sp, files=len(list((DS / sp / "images").iterdir())), no_box_files=em, **{c + "_boxes": bx[c] for c in CLASSES}))
print(pd.DataFrame(rows).to_string(index=False))
v1_val, v1_test = len(list((V1_DATASET / "val" / "images").iterdir())), len(list((V1_DATASET / "test" / "images").iterdir()))
assert len(list((DS / "val" / "images").iterdir())) == v1_val and len(list((DS / "test" / "images").iterdir())) == v1_test
print("\n✓ validation and test are exactly the v1 photos (%d / %d)" % (v1_val, v1_test))

In [ ]:
# CELL 8 - Training Plan (read before you train)
# ─────────────────────────────────────────────────────────────────────────────
# Just prints what will happen. Training takes about 1.5 hours on the RTX 4060.
#   BEFORE YOU RUN CELL 9:  plug in the laptop, turn off sleep, keep an Ubuntu window open, close Edge.
# ─────────────────────────────────────────────────────────────────────────────
best_pt = OUT_ROOT / "runs" / RUN_NAME / "weights" / "best.pt"
print("run name   :", RUN_NAME)
print("data       :", DS / "data.yaml")
print("weights out:", best_pt)
print("already trained:", best_pt.exists(), "(if True, CELL 9 is skipped; delete the run folder to train again)")

In [ ]:
# CELL 9 - Train the Detector
# ─────────────────────────────────────────────────────────────────────────────
# YOLOv8n, same settings as v1. If the notebook was interrupted, set RESUME = True to continue from the last saved epoch.
# The best model (by validation mAP) is saved as best.pt; the notebook never uses the test photos for training or choosing.
# ─────────────────────────────────────────────────────────────────────────────
RESUME = False
last_pt = OUT_ROOT / "runs" / RUN_NAME / "weights" / "last.pt"
if best_pt.exists() and not RESUME:
    print("already trained -> skipped:", best_pt)
elif RESUME and last_pt.exists():
    YOLO(str(last_pt)).train(resume=True)
else:
    m = YOLO(MODEL_START)
    m.train(data=str(DS / "data.yaml"), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, device=0 if torch.cuda.is_available() else "cpu",
            workers=WORKERS, seed=SEED, cos_lr=True, close_mosaic=10, project=str(OUT_ROOT / "runs"), name=RUN_NAME, exist_ok=True,
            fliplr=0.5, flipud=0.0, degrees=10, scale=0.5, hsv_h=0.015, hsv_s=0.5, hsv_v=0.35)
assert best_pt.exists(), "training did not produce best.pt"
print("✓ best weights:", best_pt)

In [ ]:
# CELL 10 - Training Curves
# ─────────────────────────────────────────────────────────────────────────────
# Loss and validation score per epoch. Check: validation mAP rises, then levels off; the best epoch is marked.
# ─────────────────────────────────────────────────────────────────────────────
res = pd.read_csv(OUT_ROOT / "runs" / RUN_NAME / "results.csv"); res.columns = [c.strip() for c in res.columns]
best_ep = int(res["metrics/mAP50(B)"].idxmax())
fig, axs = plt.subplots(1, 3, figsize=(15, 3.8))
for c in ("train/box_loss", "train/cls_loss", "val/box_loss", "val/cls_loss"):
    axs[0].plot(res["epoch"], res[c], label=c.split("/")[0] + " " + c.split("/")[1].replace("_loss", ""))
axs[0].set_title("Loss"); axs[0].legend(fontsize=7); axs[0].grid(alpha=.3)
axs[1].plot(res["epoch"], res["metrics/mAP50(B)"], label="mAP@0.5", lw=2); axs[1].plot(res["epoch"], res["metrics/mAP50-95(B)"], label="mAP@0.5-0.95", lw=2)
axs[1].axvline(res["epoch"][best_ep], color="grey", ls="--"); axs[1].set_title("Validation mAP (dashed = best epoch)"); axs[1].legend(); axs[1].grid(alpha=.3)
axs[2].plot(res["epoch"], res["metrics/precision(B)"], label="precision"); axs[2].plot(res["epoch"], res["metrics/recall(B)"], label="recall")
axs[2].set_title("Validation precision / recall"); axs[2].legend(); axs[2].grid(alpha=.3)
plt.tight_layout(); fig.savefig(OUT_ROOT / "figures" / "training_curves.png", dpi=200); plt.show()
print("epochs run: %d | best epoch: %d | best val mAP50: %.3f | mAP50-95: %.3f" % (len(res), res["epoch"][best_ep], res["metrics/mAP50(B)"][best_ep], res["metrics/mAP50-95(B)"][best_ep]))

In [ ]:
# CELL 11 - Evaluation Helper
# ─────────────────────────────────────────────────────────────────────────────
# evaluate(): precision / recall / mAP for each class, for ONE model on ONE set of photos.
# compare(): the same for v1 and v3 side by side, with the change.
# ─────────────────────────────────────────────────────────────────────────────
def evaluate(weights, data_yaml, split):
    r = YOLO(str(weights)).val(data=str(data_yaml), split=split, imgsz=IMGSZ, conf=0.001, plots=False, verbose=False, workers=WORKERS, batch=BATCH)
    per = {}
    for k, ci in enumerate(r.box.ap_class_index):
        p, rc, ap50, ap = r.box.class_result(k)
        per[CLASSES[int(ci)]] = dict(precision=float(p), recall=float(rc), mAP50=float(ap50), mAP50_95=float(ap))
    per["ALL"] = dict(precision=float(r.box.mp), recall=float(r.box.mr), mAP50=float(r.box.map50), mAP50_95=float(r.box.map))
    return per

def compare(data_yaml, split, title):
    a, b = evaluate(V1_MODEL, data_yaml, split), evaluate(best_pt, data_yaml, split)
    rows = [dict(cls=c, metric=m_, v1=round(a[c][m_], 3), v3=round(b[c][m_], 3), change=round(b[c][m_] - a[c][m_], 3))
            for c in CLASSES + ["ALL"] if c in a and c in b for m_ in ("precision", "recall", "mAP50", "mAP50_95")]
    df = pd.DataFrame(rows)
    print("\n=== %s ===" % title)
    print(df.pivot(index="metric", columns="cls", values=["v1", "v3", "change"]).round(3).to_string())
    return df, a, b
print("✓ evaluation helpers ready")

In [ ]:
# CELL 12 - Original Test Photos: v1 vs v3  (human-drawn boxes from the public datasets)
# ─────────────────────────────────────────────────────────────────────────────
# The same 566 test photos as v1. Check: v3 should not be clearly WORSE here (a change under about 0.02 mAP50 is noise).
# Eczema has only 32 test photos, so its numbers move a lot.
# ─────────────────────────────────────────────────────────────────────────────
df_orig, a_orig, b_orig = compare(DS / "data.yaml", "test", "ORIGINAL 566 TEST PHOTOS")
df_orig.to_csv(OUT_ROOT / "reports" / "compare_original_test.csv", index=False)
d_orig = b_orig["ALL"]["mAP50"] - a_orig["ALL"]["mAP50"]
print("\nmAP50 change on the original test photos: %+.3f" % d_orig)

In [ ]:
# CELL 13 - Your Hand-Boxed Test Photos: v1 vs v3
# ─────────────────────────────────────────────────────────────────────────────
# The honest test: photos like yours, boxed by people, never used for training. Skipped if OWN_TEST_DIR is None.
# ─────────────────────────────────────────────────────────────────────────────
d_own = None
if OWN_TEST_DIR and (Path(OWN_TEST_DIR) / "images").exists():
    yaml.safe_dump({"path": str(OWN_TEST_DIR), "train": "images", "val": "images", "test": "images", "names": {i: c for i, c in enumerate(CLASSES)}}, open(Path(OWN_TEST_DIR) / "data.yaml", "w"))
    df_own, a_own, b_own = compare(Path(OWN_TEST_DIR) / "data.yaml", "val", "YOUR HAND-BOXED PHOTOS")
    df_own.to_csv(OUT_ROOT / "reports" / "compare_own_test.csv", index=False)
    d_own = b_own["ALL"]["mAP50"] - a_own["ALL"]["mAP50"]
    print("\nmAP50 change on your hand-boxed photos: %+.3f" % d_own)
else:
    print("no hand-boxed test set set in CELL 2: skipped")

In [ ]:
# CELL 14 - Healthy Skin: How Many False Boxes?
# ─────────────────────────────────────────────────────────────────────────────
# Runs both models on the healthy photos that were kept OUT of training and counts boxes drawn at the app's confidence limits.
# Fewer boxes = fewer false alarms on healthy skin. (The photos contain no lesions, so every box here is a false alarm.)
# ─────────────────────────────────────────────────────────────────────────────
def false_alarms(weights, paths):
    m = YOLO(str(weights)); per_photo = []
    for i in range(0, len(paths), 32):
        for r in m.predict(paths[i:i + 32], imgsz=IMGSZ, conf=min(APP_THRESHOLDS.values()), iou=0.45, verbose=False, device=0 if torch.cuda.is_available() else "cpu"):
            n = 0
            if r.boxes is not None and len(r.boxes):
                for c, cf in zip(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                    n += cf >= APP_THRESHOLDS[CLASSES[int(c)]]
            per_photo.append(int(n))
    a = np.array(per_photo)
    return dict(photos=len(a), mean_boxes_per_photo=round(float(a.mean()), 3), photos_with_any_box_pct=round(100 * float((a > 0).mean()), 1))
fa1, fa3 = false_alarms(V1_MODEL, HOLDOUT_NORMAL), false_alarms(best_pt, HOLDOUT_NORMAL)
print(pd.DataFrame({"v1": fa1, "v3": fa3}).to_string())

In [ ]:
# CELL 15 - Prediction Gallery (look at real boxes)
# ─────────────────────────────────────────────────────────────────────────────
# 12 random test photos with the v3 boxes drawn at the app's confidence limits. Check by eye: do the boxes sit on real lesions?
# ─────────────────────────────────────────────────────────────────────────────
tfiles = sorted((DS / "test" / "images").iterdir()); random.Random(1).shuffle(tfiles)
m3 = YOLO(str(best_pt))
fig, axs = plt.subplots(3, 4, figsize=(18, 13))
for ax, f in zip(axs.ravel(), tfiles[:12]):
    r = m3.predict(str(f), imgsz=IMGSZ, conf=min(APP_THRESHOLDS.values()), verbose=False, device=0 if torch.cuda.is_available() else "cpu")[0]
    keep = [i for i, (c, cf) in enumerate(zip(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy())) if cf >= APP_THRESHOLDS[CLASSES[int(c)]]]
    r.boxes = r.boxes[keep] if len(keep) else r.boxes[:0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.tight_layout(); fig.savefig(OUT_ROOT / "figures" / "prediction_gallery.png", dpi=150); plt.show()

In [ ]:
# CELL 16 - ONNX Export & Verification
# ─────────────────────────────────────────────────────────────────────────────
# Converts best.pt to ONNX for the C# app (640 x 640 fixed input, opset 12), measures CPU speed, and checks that the ONNX file finds the
# same number of boxes as the PyTorch model on 20 test photos (counted at confidence 0.25).
# The file is saved as lesion_detector_v3_...onnx - the v1 files in the app are NOT touched.
# ─────────────────────────────────────────────────────────────────────────────
import onnxruntime as ort
onnx_src = Path(YOLO(str(best_pt)).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
onnx_path = EXPORT_DIR / ("lesion_%s_imgsz%d.onnx" % (RUN_NAME, IMGSZ))
shutil.copy2(onnx_src, onnx_path)
so = ort.SessionOptions(); so.intra_op_num_threads = 4
sess = ort.InferenceSession(str(onnx_path), so, providers=["CPUExecutionProvider"]); inp = sess.get_inputs()[0]
x = np.random.rand(1, 3, IMGSZ, IMGSZ).astype("float32")
for _ in range(3): sess.run(None, {inp.name: x})
ts = []
for _ in range(20):
    t0 = time.perf_counter(); out = sess.run(None, {inp.name: x}); ts.append((time.perf_counter() - t0) * 1000)
LATENCY = {"median_ms": round(float(np.median(ts)), 1), "p95_ms": round(float(np.percentile(ts, 95)), 1), "threads": 4}
print("ONNX saved:", onnx_path.name, "| CPU latency:", LATENCY, "| output shape:", out[0].shape)

pt_model, onnx_model = YOLO(str(best_pt)), YOLO(str(onnx_path), task="detect")
agree, tot = 0, 0
for f in tfiles[:20]:
    n1 = len(pt_model.predict(str(f), imgsz=IMGSZ, conf=0.25, verbose=False, device="cpu")[0].boxes)
    n2 = len(onnx_model.predict(str(f), imgsz=IMGSZ, conf=0.25, verbose=False)[0].boxes)
    agree += (abs(n1 - n2) <= 1); tot += 1
print("ONNX vs PyTorch: box counts agree (within 1) on %d of %d photos" % (agree, tot))

In [ ]:
# CELL 17 - Decision and Records
# ─────────────────────────────────────────────────────────────────────────────
# Prints the verdict and saves everything for your documentation: settings, results, checksum, latency, verdict.
# RULE: v3 is worth using only if it is clearly better on YOUR photos (or healthy skin) AND not worse on the original test photos.
# Putting v3 into the app is a separate step (copy the ONNX/JSON as lesion_detector_v1.*, rebuild, re-tune the thresholds on validation).
# ─────────────────────────────────────────────────────────────────────────────
better_own = d_own is not None and d_own >= 0.02
fewer_false = fa3["mean_boxes_per_photo"] <= 0.8 * fa1["mean_boxes_per_photo"] if fa1["mean_boxes_per_photo"] > 0 else False
not_worse = d_orig >= -0.01
if (better_own or fewer_false) and not_worse:
    VERDICT = "v3 is better (%s) and not worse on the original test: worth using." % ("on your photos" if better_own else "fewer false boxes on healthy skin")
elif d_orig <= -0.03 or (d_own is not None and d_own <= -0.02):
    VERDICT = "v3 is WORSE: keep v1."
else:
    VERDICT = "no clear difference: keep v1 (already deployed and verified)."
print("VERDICT:", VERDICT)

meta = dict(model="PrecisionSkin lesion detector v3 (own boxes)", created=datetime.datetime.now().isoformat(timespec="seconds"), classes=CLASSES, imgsz=IMGSZ,
            recipe=dict(start=MODEL_START, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, seed=SEED), own_box_sources=[s["name"] for s in OWN_BOX_SOURCES],
            own_hand_boxed_added=n_added, own_repeat=OWN_REPEAT, normal_background_train=len(trainbg), normal_holdout=len(hold),
            original_test_change_mAP50=round(float(d_orig), 4), own_test_change_mAP50=(round(float(d_own), 4) if d_own is not None else None),
            false_boxes_healthy_skin=dict(v1=fa1, v3=fa3), cpu_latency=LATENCY, onnx_sha256=hashlib.sha256(onnx_path.read_bytes()).hexdigest(), verdict=VERDICT,
            note="Research prototype. Boxes show candidate lesion locations; not a diagnosis.")
json.dump(meta, open(EXPORT_DIR / (onnx_path.stem + ".json"), "w"), indent=2, default=str)
json.dump(meta, open(OUT_ROOT / "reports" / "summary.json", "w"), indent=2, default=str)
print("saved:", EXPORT_DIR / (onnx_path.stem + ".json"))

In [ ]:
# CELL 18 - Final Sanity Assertions
# ─────────────────────────────────────────────────────────────────────────────
# Quick checks that nothing was mixed up. All lines should print ✓.
# ─────────────────────────────────────────────────────────────────────────────
assert onnx_path.exists() and (EXPORT_DIR / (onnx_path.stem + ".json")).exists(); print("✓ ONNX and JSON saved")
assert [YOLO(str(best_pt)).names[i] for i in range(3)] == CLASSES; print("✓ class order of the new model:", CLASSES)
assert len(list((DS / "test" / "images").iterdir())) == len(list((V1_DATASET / "test" / "images").iterdir())); print("✓ test photos identical to v1")
assert not any(p.name.startswith("own_") for p in (DS / "test" / "images").iterdir()); print("✓ none of your training photos is in the test set")
assert V1_MODEL.exists(); print("✓ v1 model untouched at", V1_MODEL)
print("\nVERDICT:", VERDICT)